In [6]:
%pip install -U langchain
%pip install -U langchain-openai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [23]:
import os
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from typing import List, Literal, Optional
from pydantic import BaseModel, Field


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [9]:
model = init_chat_model("gpt-4.1")

In [10]:
response = model.invoke("Why do parrots talk?")

In [12]:
print(response.content)

Parrots "talk" because they are **excellent mimics of sounds**, including human speech. Here’s **why parrots talk**:

### 1. **Highly Developed Vocal Abilities**
- Parrots have specialized vocal organs (the **syrinx**) and flexible tongues that let them reproduce a wide range of sounds.
- Many parrot species are some of the best vocal learners in the animal kingdom.

### 2. **Social Communication**
- In the wild, parrots live in flocks and use vocalizations to communicate with each other—identifying family, signaling danger, or bonding.
- Imitation is a key part of their natural social behavior.

### 3. **Adaptation to Captivity**
- When kept as pets, parrots consider humans part of their "flock."
- They mimic human speech and environmental sounds as a way to bond with and interact with their human "flock-mates."

### 4. **Attention and Enrichment**
- Parrots are **intelligent and curious**. Talking is mentally stimulating and earns them attention, treats, or interaction.
- Parrots can

# Input Schemas

In [16]:
class RunnerProfile(BaseModel):
    name: str
    age: int
    sex: Literal["male", "female", "other"]
    experience_level: Literal["beginner", "intermediate", "advanced"]
    weekly_mileage_km: float = Field(..., description="Current average weekly mileage in km")
    preferred_units: Literal["km", "mi"] = "km"
    available_days: List[Literal["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]]
    constraints: List[str] = Field(
        default_factory=list,
        description="Injuries, schedule constraints, etc"
    )

class RecentRun(BaseModel):
    date: str
    distance_km: float
    duration_min: float
    avg_pace_min_per_km: float
    notes: Optional[str] = None

# Output Schemas

In [39]:
class Workout(BaseModel):
    day: Literal["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    focus: Literal["easy", "long_run", "intervals", "tempo", "recovery", "rest"]
    distance_km: float = Field(..., description="Planned distance in km (0 if rest)")
    target_pace_min_per_km: Optional[float] = Field(
        None, description="Leave None for recovery or rest workouts."
    )
    description: str = Field(..., description="Plain-language description of the workout")
    notes: Optional[str] = None

class WeeklyPlan(BaseModel):
    week_number: int = Field(..., description="Week number in the overall plan")
    focus_summary: str = Field(..., description="High-level weekly focus or theme")
    workouts: List[Workout] = Field(..., description="List of daily workouts for this week")

class TrainingPlan(BaseModel):
    goal_description: str = Field(..., description="High-level description of the overall goal")
    plan_duration_weeks: int = Field(..., description="Total number of weeks in the plan")
    weekly_overview: str = Field(..., description="Summary of how training load evolves week by week")
    weekly_plans: List[WeeklyPlan] = Field(..., description="List of structured weekly plans")

# Chaining

In [40]:
# Base LLM (swap for another provider/model if you like)
llm = ChatOpenAI(
    model="gpt-4o-mini",   # or gpt-4o, etc.
    temperature=0.7,
)

# Tell LangChain that this LLM should return a TrainingPlan
plan_llm = llm.with_structured_output(TrainingPlan)


In [41]:
system_msg = """
You are RunBuddy, an AI running coach.

You must:
- Be conservative about sudden mileage increases.
- Respect injuries and constraints.
- Use the runner's preferred units for wording (km vs miles).
- Align the workouts with their available days only.
- Wherever possible, include pace.

You answer STRICTLY by filling the TrainingPlan schema. Do not add extra keys.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_msg),
        (
            "human",
            """Create a {weeks}-week training plan.

Runner profile:
{runner_profile}

Recent runs (last 2–4 weeks):
{recent_runs}

Goal:
{goal_description}
"""
        ),
    ]
)

# Combine: prompt -> structured LLM
plan_chain = prompt | plan_llm


# Simulating output

In [42]:
runner = RunnerProfile(
    name="Joel",
    age=24,
    sex="male",
    experience_level="intermediate",
    weekly_mileage_km=35.0,
    preferred_units="km",
    available_days=["Mon", "Wed", "Thu", "Sat", "Sun"],
    constraints=["Mild knee pain if running >18km long runs", "Busy on Tuesdays"],
)

recent_runs = [
    RecentRun(
        date="2025-10-28",
        distance_km=10.2,
        duration_min=54.0,
        avg_pace_min_per_km=5.29,
        rpe=7,
        notes="Felt strong but last 2km were hard"
    ),
    RecentRun(
        date="2025-10-30",
        distance_km=6.0,
        duration_min=34.0,
        avg_pace_min_per_km=5.67,
        rpe=5,
        notes="Easy neighborhood run"
    ),
    RecentRun(
        date="2025-11-02",
        distance_km=14.0,
        duration_min=80.0,
        avg_pace_min_per_km=5.71,
        rpe=8,
        notes="Long run, knee a bit sore in last 3km"
    ),
]

goal_description = "Run a sub-50-minute 10K race in 10 weeks."


In [46]:
plan: TrainingPlan = plan_chain.invoke(
    {
        "weeks": 24,
        "runner_profile": runner.model_dump(),
        "recent_runs": [r.model_dump() for r in recent_runs],
        "goal_description": goal_description,
    }
)

plan


TrainingPlan(goal_description='Run a sub-50-minute 10K race in 10 weeks.', plan_duration_weeks=24, weekly_overview='Gradually increase mileage and intensity, focusing on speed and endurance leading up to the 10K race in week 10, followed by recovery and base building in subsequent weeks.', weekly_plans=[WeeklyPlan(week_number=1, focus_summary='Base building with easy runs', workouts=[Workout(day='Mon', focus='easy', distance_km=6.0, target_pace_min_per_km=5.5, description='Easy run to start the week', notes=None), Workout(day='Wed', focus='easy', distance_km=8.0, target_pace_min_per_km=5.5, description='Maintain a comfortable pace', notes=None), Workout(day='Thu', focus='recovery', distance_km=4.0, target_pace_min_per_km=None, description='Recovery run, keep it light', notes=None), Workout(day='Sat', focus='long_run', distance_km=12.0, target_pace_min_per_km=6.0, description='Long run, monitor knee pain', notes='Do not exceed 18km.'), Workout(day='Sun', focus='rest', distance_km=0.0, t

In [47]:
plan.weekly_plans

[WeeklyPlan(week_number=1, focus_summary='Base building with easy runs', workouts=[Workout(day='Mon', focus='easy', distance_km=6.0, target_pace_min_per_km=5.5, description='Easy run to start the week', notes=None), Workout(day='Wed', focus='easy', distance_km=8.0, target_pace_min_per_km=5.5, description='Maintain a comfortable pace', notes=None), Workout(day='Thu', focus='recovery', distance_km=4.0, target_pace_min_per_km=None, description='Recovery run, keep it light', notes=None), Workout(day='Sat', focus='long_run', distance_km=12.0, target_pace_min_per_km=6.0, description='Long run, monitor knee pain', notes='Do not exceed 18km.'), Workout(day='Sun', focus='rest', distance_km=0.0, target_pace_min_per_km=None, description='Rest day for recovery', notes=None)]),
 WeeklyPlan(week_number=2, focus_summary='Introduce interval training', workouts=[Workout(day='Mon', focus='easy', distance_km=7.0, target_pace_min_per_km=5.5, description='Easy run to recover from long run', notes=None), Wo

In [48]:
def print_plan(plan):
    for week in plan.weekly_plans:
        print(f"\n=== Week {week.week_number}: {week.focus_summary} ===")
        for w in week.workouts:
            pace = f"{w.target_pace_min_per_km:.1f} min/km" if w.target_pace_min_per_km else "Easy / not specified"
            print(
                f"- {w.day:<3} | {w.focus:<10} | "
                f"{w.distance_km:>4.1f} km | {pace}\n"
                f"    {w.description}"
            )
            if w.notes:
                print(f"    Notes: {w.notes}")

print_plan(plan)



=== Week 1: Base building with easy runs ===
- Mon | easy       |  6.0 km | 5.5 min/km
    Easy run to start the week
- Wed | easy       |  8.0 km | 5.5 min/km
    Maintain a comfortable pace
- Thu | recovery   |  4.0 km | Easy / not specified
    Recovery run, keep it light
- Sat | long_run   | 12.0 km | 6.0 min/km
    Long run, monitor knee pain
    Notes: Do not exceed 18km.
- Sun | rest       |  0.0 km | Easy / not specified
    Rest day for recovery

=== Week 2: Introduce interval training ===
- Mon | easy       |  7.0 km | 5.5 min/km
    Easy run to recover from long run
- Wed | intervals  |  6.0 km | 4.5 min/km
    4x800m intervals at a fast pace
    Notes: Ensure to warm up and cool down.
- Thu | recovery   |  4.0 km | Easy / not specified
    Light recovery run
- Sat | long_run   | 14.0 km | 6.0 min/km
    Increase distance for long run, monitor knee pain
- Sun | rest       |  0.0 km | Easy / not specified
    Rest day

=== Week 3: Tempo run introduction ===
- Mon | easy     

# Agents

Main planners
1. Planning agent
2. Nutrition agent
3. Hydration agent

Revisors
1. Plan revision agent
2. Safety agent

System
1. Logging agent
2. Notification agent